# Carbon Footprint Modeling using Taylor Series Expansion
## Emission Sensitivity Analysis

**Course:** Mathematics for Intelligent Systems 2 (25MAT116)  
**Topic:** Taylor Series for Carbon Footprint Modeling  

---

### Mathematical Framework

The carbon emission function is defined as:

$$C(x) = \sum_{i=1}^{7} e_i x_i + \frac{1}{2}\sum_{i=1}^{7} a_i x_i^2 + \frac{1}{2} x^T B x$$

The Taylor series expansion around baseline $x_0$:

$$C(x) \approx C(x_0) + \nabla C(x_0) \cdot (x - x_0) + \frac{1}{2}(x-x_0)^T H(x_0)(x-x_0)$$

The normalized sensitivity index:

$$S_i = \frac{\partial C}{\partial x_i} \cdot \frac{x_i}{C}$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

from carbon_model import CarbonFootprintModel, BASELINE_ACTIVITIES, EMISSION_FACTORS

# Initialize model
model = CarbonFootprintModel()
x0 = np.array(list(BASELINE_ACTIVITIES.values()))
e = np.array(list(EMISSION_FACTORS.values()))

model.set_parameters(e, np.zeros(7), np.zeros(21), x0)

C_baseline = model.compute_emissions(x0)
contributions = e * x0

print(f'Model initialized successfully')
print(f'Total baseline emissions: {C_baseline:.2f} Gt CO2e')

## 1. Baseline Emissions Breakdown

In [ ]:
df_baseline = pd.DataFrame({
    'Sector': model.var_names,
    'Emissions_Gt': contributions,
    'Percentage': (contributions / C_baseline) * 100
}).sort_values('Emissions_Gt', ascending=False)

print('Baseline Emissions Breakdown:')
print('='*55)
print(df_baseline.to_string(index=False))
print('='*55)
print(f"TOTAL: {C_baseline:.2f} Gt CO2e")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
colors = plt.cm.Set2(np.linspace(0, 1, 7))

# Pie chart
ax1.pie(df_baseline['Emissions_Gt'], labels=df_baseline['Sector'],
        autopct='%1.1f%%', colors=colors, startangle=90)
ax1.set_title('Global Emissions by Sector\n(Gt CO2e)', fontsize=13, fontweight='bold')

# Bar chart
ax2.barh(df_baseline['Sector'], df_baseline['Emissions_Gt'], color=colors)
ax2.set_xlabel('Emissions (Gt CO2e)', fontsize=12)
ax2.set_title('Emissions by Sector', fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.4)
for i, v in enumerate(df_baseline['Emissions_Gt']):
    ax2.text(v + 0.1, i, f'{v:.2f}', va='center', fontsize=10)

plt.suptitle('Baseline Global Carbon Emissions (2024-2025)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_baseline_breakdown.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Gradient Vector and Sensitivity Analysis

In [ ]:
df_sensitivity = model.sensitivity_analysis(perturbation_pct=10)

print('Sensitivity Analysis Results (10% perturbation):')
print('='*70)
print(df_sensitivity[['Sector','Gradient','Sensitivity_Index','Absolute_Impact_Gt','Percent_Impact']].to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Sensitivity index bar chart
bars = ax1.barh(df_sensitivity['Sector'][::-1],
                df_sensitivity['Sensitivity_Index'][::-1],
                color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 7)))
ax1.set_xlabel('Normalized Sensitivity Index (Si)', fontsize=12)
ax1.set_title('Parameter Sensitivity Ranking', fontsize=13, fontweight='bold')
ax1.grid(axis='x', alpha=0.4)

# Tornado diagram
positive = df_sensitivity['Absolute_Impact_Gt'].values[::-1]
negative = -positive
sectors = df_sensitivity['Sector'].values[::-1]
y = np.arange(len(sectors))

ax2.barh(y, positive, color='#e74c3c', alpha=0.8, label='+10%')
ax2.barh(y, negative, color='#3498db', alpha=0.8, label='-10%')
ax2.set_yticks(y)
ax2.set_yticklabels(sectors)
ax2.axvline(x=0, color='black', linewidth=0.8)
ax2.set_xlabel('Change in Emissions (Gt CO2e)', fontsize=12)
ax2.set_title('Tornado Diagram (±10% perturbation)', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(axis='x', alpha=0.4)

plt.suptitle('Emission Sensitivity Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_sensitivity_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Taylor Approximation Accuracy

In [ ]:
perturbations = np.linspace(-25, 25, 60)
actual, taylor1, taylor2 = [], [], []

for p in perturbations:
    x_test = x0 * (1 + p/100)
    actual.append(model.compute_emissions(x_test))
    taylor1.append(model.taylor_first_order(x_test, x0))
    taylor2.append(model.taylor_second_order(x_test, x0))

actual = np.array(actual)
taylor1 = np.array(taylor1)
taylor2 = np.array(taylor2)

# R-squared calculation
ss_tot = np.sum((actual - actual.mean())**2)
r2_1 = 1 - np.sum((actual - taylor1)**2) / ss_tot
r2_2 = 1 - np.sum((actual - taylor2)**2) / ss_tot

print(f'1st-order Taylor R²: {r2_1:.6f}')
print(f'2nd-order Taylor R²: {r2_2:.6f}')
print(f'Maximum 1st-order error: {np.max(np.abs(actual - taylor1)):.6f} Gt')
print(f'Maximum 2nd-order error: {np.max(np.abs(actual - taylor2)):.6f} Gt')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(perturbations, actual, 'k-', linewidth=2.5, label='Actual C(x)')
ax1.plot(perturbations, taylor1, 'b--', linewidth=2, label=f'1st Order (R²={r2_1:.4f})')
ax1.plot(perturbations, taylor2, 'r:', linewidth=2, label=f'2nd Order (R²={r2_2:.4f})')
ax1.set_xlabel('Uniform Perturbation (%)', fontsize=12)
ax1.set_ylabel('Total Emissions (Gt CO2e)', fontsize=12)
ax1.set_title('Taylor Approximation vs Actual', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.4)

error1 = np.abs(actual - taylor1)
error2 = np.abs(actual - taylor2)
ax2.plot(perturbations, error1, 'b-', linewidth=2, label='1st Order Error')
ax2.plot(perturbations, error2, 'r-', linewidth=2, label='2nd Order Error')
ax2.set_xlabel('Uniform Perturbation (%)', fontsize=12)
ax2.set_ylabel('Absolute Error (Gt CO2e)', fontsize=12)
ax2.set_title('Approximation Error', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.4)

plt.suptitle('Taylor Series Approximation Validation', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_taylor_validation.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Scenario Analysis

In [ ]:
scenarios = {
    'Baseline': x0.copy(),
    'Scenario_A': x0.copy(),
    'Scenario_B': x0.copy(),
    'Aggressive': x0.copy(),
}
scenario_labels = [
    'Baseline',
    'Scenario A\n(10% top 3)',
    'Scenario B\n(20% Elec + 10% Trans)',
    'Aggressive\n(20% all sectors)'
]

scenarios['Scenario_A'][0] *= 0.9
scenarios['Scenario_A'][1] *= 0.9
scenarios['Scenario_A'][2] *= 0.9

scenarios['Scenario_B'][0] *= 0.8
scenarios['Scenario_B'][1] *= 0.9

scenarios['Aggressive'] *= 0.8

df_scenarios = model.scenario_analysis(scenarios)
df_scenarios['Label'] = scenario_labels

print('Scenario Analysis Results:')
print('='*70)
print(df_scenarios[['Label','Total_Emissions_Gt','Reduction_Gt','Reduction_Pct']].to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
colors = ['#95a5a6', '#3498db', '#e74c3c', '#2ecc71']

bars = ax1.bar(df_scenarios['Label'], df_scenarios['Total_Emissions_Gt'],
               color=colors, alpha=0.85)
ax1.axhline(y=C_baseline, color='black', linestyle='--', linewidth=1.5, label='Baseline')
ax1.set_ylabel('Total Emissions (Gt CO2e)', fontsize=12)
ax1.set_title('Total Emissions by Scenario', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.4)
for bar in bars:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., h + 0.2,
             f'{h:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax2.barh(df_scenarios['Label'][1:], df_scenarios['Reduction_Gt'][1:],
         color=colors[1:], alpha=0.85)
ax2.set_xlabel('Emission Reduction (Gt CO2e)', fontsize=12)
ax2.set_title('Reduction Achieved per Scenario', fontsize=13, fontweight='bold')
ax2.grid(axis='x', alpha=0.4)

plt.suptitle('Emission Reduction Scenario Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_scenario_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Summary and Key Findings

In [ ]:
print('='*70)
print('CARBON FOOTPRINT ANALYSIS - KEY FINDINGS')
print('='*70)
print(f'\nTotal Global Emissions:          {C_baseline:.2f} Gt CO2e')
print(f'Highest Sensitivity Sector:      {df_sensitivity.iloc[0]["Sector"]}')
print(f'Top 3 Sectors Sensitivity Share: {df_sensitivity.head(3)["Percent_Impact"].sum():.1f}%')
print(f'Taylor Approximation R²:         {r2_1:.6f}')
print()
print('PRIORITY RECOMMENDATIONS:')
for i, row in df_sensitivity.head(3).iterrows():
    print(f'  {i+1}. {row["Sector"]:<25} (Si = {row["Sensitivity_Index"]:.4f})')
print()
print('SCENARIO IMPACT:')
for _, row in df_scenarios.iterrows():
    if row['Reduction_Gt'] > 0:
        print(f'  {row["Label"].replace(chr(10), " "):<40} → {row["Reduction_Gt"]:.2f} Gt saved ({row["Reduction_Pct"]:.1f}%)')
print('='*70)

In [ ]:
# Export all results
df_sensitivity.to_csv('sensitivity_results.csv', index=False)
df_scenarios.to_csv('scenarios_results.csv', index=False)

pd.DataFrame({
    'Sector': model.var_names,
    'Emission_Factor': e,
    'Baseline_Activity': x0,
    'Emissions_Gt': contributions,
    'Gradient': model.compute_gradient(x0),
    'Sensitivity_Index': model.compute_gradient(x0) * (x0 / C_baseline)
}).to_csv('validation_results.csv', index=False)

print('✓ sensitivity_results.csv')
print('✓ scenarios_results.csv')
print('✓ validation_results.csv')
print('✓ plot_baseline_breakdown.png')
print('✓ plot_sensitivity_analysis.png')
print('✓ plot_taylor_validation.png')
print('✓ plot_scenario_comparison.png')
print('\nAll outputs generated successfully!')